## Archived Notebook

This notebook has been superseded by `notebooks/02_validate_with_gx_framework.ipynb`.

Keep this file only as historical reference for the earlier `dq` wrapper demo flow.

# DQ Wrapper Notebook Demo

This notebook demonstrates importing and using the function-first Great Expectations wrapper with deterministic suite discovery.

In [1]:
from pathlib import Path
import importlib
import sys

cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Ensure notebook uses the in-repo dq package from src/.
for name in [mod for mod in list(sys.modules) if mod == "dq" or mod.startswith("dq.")]:
    sys.modules.pop(name)

run_data_quality = importlib.import_module("dq.dq_runner").run_data_quality
build_suite_name = importlib.import_module("dq.dq_runner").build_suite_name
resolve_suite_path = importlib.import_module("dq.dq_runner").resolve_suite_path

print("Repository root:", repo_root)
print("dq module:", run_data_quality.__module__)
print("Imports successful")

Repository root: /workspaces/great-expectations
dq module: dq.dq_runner
Imports successful


In [2]:
from pathlib import Path
import shutil

import pandas as pd
from deltalake import DeltaTable, write_deltalake

csv_path = repo_root / "data" / "green_tripdata_2017_sample.csv"
delta_path = repo_root / "data" / "green_tripdata_2017_sample.delta"

if delta_path.exists():
    shutil.rmtree(delta_path)

pdf = pd.read_csv(csv_path)
write_deltalake(str(delta_path), pdf, mode="overwrite")

delta_table = DeltaTable(str(delta_path))
print("Created Delta test dataset:", delta_path)
print("Delta version:", delta_table.version())
print("Row count:", len(delta_table.to_pandas()))

Created Delta test dataset: /workspaces/great-expectations/data/green_tripdata_2017_sample.delta
Delta version: 0
Row count: 10


In [3]:
from deltalake import DeltaTable
from pyspark.sql import SparkSession

spark = SparkSession.getActiveSession() or (
    SparkSession.builder
    .appName("dq-wrapper-notebook-demo")
    .master("local[*]")
    .getOrCreate()
)

delta_source_path = repo_root / "data" / "green_tripdata_2017_sample.delta"
if not delta_source_path.exists():
    raise FileNotFoundError(
        f"Delta test dataset not found at {delta_source_path}. "
        "Run Cell 3 first to create it."
    )

# Load Delta via deltalake and convert through PyArrow to avoid pandas->Spark distutils issues.
delta_table = DeltaTable(str(delta_source_path))
arrow_table = delta_table.to_pyarrow_table()
rows = arrow_table.to_pylist()
if not rows:
    raise RuntimeError(f"Delta dataset at {delta_source_path} has no rows")

df = spark.createDataFrame(rows).limit(1000)

source_format = "delta"
source_path = delta_source_path

print("Delta dataset loaded for GX validation")
print(f"Source format: {source_format}")
print(f"Source path: {source_path}")
print(f"Rows loaded: {df.count()}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 18:03:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Delta dataset loaded for GX validation
Source format: delta
Source path: /workspaces/great-expectations/data/green_tripdata_2017_sample.delta


Rows loaded: 10


In [4]:
layer = "bronze"
schema_name = "sales"
table_name = "green_tripdata_2017"

suite_name = build_suite_name(layer, schema_name, table_name)
suite_path = resolve_suite_path(
    layer,
    schema_name,
    table_name,
    project_root=repo_root,
)

print("Derived suite name:", suite_name)
print("Expected suite path:", suite_path)

Derived suite name: bronze.sales.green_tripdata_2017
Expected suite path: /workspaces/great-expectations/gx/expectations/bronze.sales.green_tripdata_2017.yml


In [5]:
dq_metrics_path = repo_root / "logs" / "dq_metrics.delta"

dq_metrics_path.parent.mkdir(parents=True, exist_ok=True)

result = run_data_quality(
    df=df,
    layer=layer,
    schema_name=schema_name,
    table_name=table_name,
    log_metrics=True,
    storage_profile="local_fs",
    source_format=source_format,
    metrics_sink="path",
    metrics_format="delta",
    dq_metrics_path=str(dq_metrics_path),
    project_root=repo_root,
)

result

Calculating Metrics: 100%|██████████| 23/23 [00:00<00:00, 28.85it/s]


{'suite_name': 'bronze.sales.green_tripdata_2017',
 'run_id': 'b175b129-d9d4-45c0-9d0d-dbf0fd2fd6dd',
 'success': True,
 'evaluated_expectations': 3,
 'successful_expectations': 3,
 'failed_expectations': 0,
 'sampling_strategy': 'full',
 'original_row_count': 10,
 'sample_row_count': 10,
 'confidence': 0.95,
 'margin_error': 0.01,
 'stratify_by': None,
 'metrics_logged': True,
 'metrics_rows_written': 3,
 'dq_metrics_table': 'analytics.dq_metrics',
 'dq_metrics_path': '/workspaces/great-expectations/logs/dq_metrics.delta',
 'metrics_sink': 'path',
 'metrics_format': 'delta'}

In [6]:
if result.get("success"):
    print("Validation succeeded")
    print("Run ID:", result.get("run_id"))
    print("Evaluated expectations:", result.get("evaluated_expectations"))
    print("Failed expectations:", result.get("failed_expectations"))
    print("Metrics logged:", result.get("metrics_logged"))
    print("Metrics rows written:", result.get("metrics_rows_written"))
    print("Metrics sink:", result.get("metrics_sink"))
    print("Metrics format:", result.get("metrics_format"))
    print("Metrics path:", result.get("dq_metrics_path"))
else:
    print("Validation failed")
    print("Error type:", result.get("error_type"))
    print(result.get("error_message"))
    print("Tip: use Python 3.10-3.13 kernel for Great Expectations local runs.")

Validation succeeded
Run ID: b175b129-d9d4-45c0-9d0d-dbf0fd2fd6dd
Evaluated expectations: 3
Failed expectations: 0
Metrics logged: True
Metrics rows written: 3
Metrics sink: path
Metrics format: delta
Metrics path: /workspaces/great-expectations/logs/dq_metrics.delta


In [7]:
from deltalake import DeltaTable

metrics_delta = DeltaTable(str(dq_metrics_path))
metrics_pdf = metrics_delta.to_pandas().sort_values("run_ts", ascending=False)

print("Metrics Delta path:", dq_metrics_path)
print("Total logged metric rows:", len(metrics_pdf))
print(metrics_pdf[
    [
        "run_id",
        "suite_name",
        "success",
        "expectation_type",
        "unexpected_count",
        "unexpected_percent",
        "run_ts",
    ]
].head(10).to_string(index=False))

Metrics Delta path: /workspaces/great-expectations/logs/dq_metrics.delta
Total logged metric rows: 6
                              run_id                       suite_name  success expectation_type  unexpected_count  unexpected_percent                           run_ts
b175b129-d9d4-45c0-9d0d-dbf0fd2fd6dd bronze.sales.green_tripdata_2017     True              NaN               NaN                 NaN 2026-03-05 18:03:46.365216+00:00
b175b129-d9d4-45c0-9d0d-dbf0fd2fd6dd bronze.sales.green_tripdata_2017     True              NaN               0.0                 0.0 2026-03-05 18:03:46.365216+00:00
b175b129-d9d4-45c0-9d0d-dbf0fd2fd6dd bronze.sales.green_tripdata_2017     True              NaN               0.0                 0.0 2026-03-05 18:03:46.365216+00:00
09b713ca-dad2-4798-b515-c05927e55590 bronze.sales.green_tripdata_2017     True              NaN               NaN                 NaN 2026-03-05 17:31:24.737399+00:00
09b713ca-dad2-4798-b515-c05927e55590 bronze.sales.green_tripdata

In [ ]:
from pathlib import Path

from deltalake import DeltaTable

candidate_paths = [
    Path("logs/dq_metrics.delta"),
    Path("../logs/dq_metrics.delta"),
]

metrics_path = next((p for p in candidate_paths if p.exists()), candidate_paths[-1])
metrics_df = DeltaTable(str(metrics_path)).to_pandas().sort_values("run_ts", ascending=False)

print(f"Metrics Delta path: {metrics_path.resolve()}")
print(f"Total rows: {len(metrics_df)}")
display(metrics_df)

Metrics Delta path: /workspaces/great-expectations/logs/dq_metrics.delta
Total rows: 6


,run_id,run_ts,layer,schema_name,table_name,suite_name,expectation_type,success,unexpected_percent,unexpected_count,element_count,sampling_strategy,original_row_count,sample_row_count,confidence,margin_error,stratify_by,details_json
0,b175b129-d9d4-45c0-9d0d-dbf0fd2fd6dd,2026-03-05 18:03:46.365216+00:00,bronze,sales,green_tripdata_2017,bronze.sales.green_tripdata_2017,NaN,True,NaN,NaN,NaN,full,10,10,0.95,0.01,NaN,"{""success"": true, ""expectation_config"": {""type..."
1,b175b129-d9d4-45c0-9d0d-dbf0fd2fd6dd,2026-03-05 18:03:46.365216+00:00,bronze,sales,green_tripdata_2017,bronze.sales.green_tripdata_2017,NaN,True,0.0,0.0,10.0,full,10,10,0.95,0.01,NaN,"{""success"": true, ""expectation_config"": {""type..."
2,b175b129-d9d4-45c0-9d0d-dbf0fd2fd6dd,2026-03-05 18:03:46.365216+00:00,bronze,sales,green_tripdata_2017,bronze.sales.green_tripdata_2017,NaN,True,0.0,0.0,10.0,full,10,10,0.95,0.01,NaN,"{""success"": true, ""expectation_config"": {""type..."
3,09b713ca-dad2-4798-b515-c05927e55590,2026-03-05 17:31:24.737399+00:00,bronze,sales,green_tripdata_2017,bronze.sales.green_tripdata_2017,NaN,True,NaN,NaN,NaN,full,10,10,0.95,0.01,NaN,"{""success"": true, ""expectation_config"": {""type..."
4,09b713ca-dad2-4798-b515-c05927e55590,2026-03-05 17:31:24.737399+00:00,bronze,sales,green_tripdata_2017,bronze.sales.green_tripdata_2017,NaN,True,0.0,0.0,10.0,full,10,10,0.95,0.01,NaN,"{""success"": true, ""expectation_config"": {""type..."
5,09b713ca-dad2-4798-b515-c05927e55590,2026-03-05 17:31:24.737399+00:00,bronze,sales,green_tripdata_2017,bronze.sales.green_tripdata_2017,NaN,True,0.0,0.0,10.0,full,10,10,0.95,0.01,NaN,"{""success"": true, ""expectation_config"": {""type..."


26/03/05 18:03:52 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
